# MEI Tools: Metadata and Music Feature Processing

This notebook guides you through two workflows for working with MEI (Music Encoding Initiative) files:

1. **Metadata workflow** — extract existing metadata into a CSV, edit that CSV to add or correct information, then apply the updated metadata back to a corpus of MEI files.
2. **Music feature workflow** — correct common encoding issues (editorial accidentals, incipits, empty verses, etc.).

The tools support MEI files produced by four common applications: **MuseScore**, **Sibelius** (via the sibmei plug-in), **Humdrum/Verovio**, and **mei-friend**.  Files are detected automatically — you do not need to specify the source application.

> **CRIM Project users:** see Section 4 for a dedicated workflow that uses the CRIM metadata spreadsheet format.

---

### Installation

Install the package directly from GitHub:

```
pip install git+https://github.com/RichardFreedman/mei_tools.git
```


---
## 1. Import Libraries

Run this cell before anything else.


In [28]:
from mei_tools import MEI_Metadata_Extractor
from mei_tools import MEI_Metadata_Updater_Generic
from mei_tools import MEI_Music_Feature_Processor
import glob
import os

---
## 2. Extract Metadata from Your MEI Files

The first step is to read your MEI files and pull out whatever metadata they already contain.  The result is a **CSV file** — one row per MEI file — that you can open in Excel, Google Sheets, or any spreadsheet application.

Each source application (MuseScore, Sibelius, Humdrum, mei-friend) stores metadata in different locations inside the MEI header.  The extractor finds the right location automatically and collects everything into a single consistent table.

> **Sibelius users:** Sibelius exports UTF-16 encoded files, which are not directly readable by most tools.  The extractor converts them to UTF-8 automatically — you do not need to do anything special.


### 2a. Specify your folders

- `mei_input_folder` — the folder containing the MEI files you want to extract metadata from.
- `csv_output_folder` — the folder where the extracted CSV file(s) will be saved.

Use an absolute path (e.g. `/Users/yourname/Documents/my_corpus`) or a path relative to this notebook.


In [29]:
# Folder containing your MEI files
mei_input_folder = 'mei_to_process'

# Folder where extracted CSV files will be written
csv_output_folder = 'extracted_metadata_csv'

### 2b. Run the extractor

One CSV file is created for each source type found in your folder (e.g. `muse_score_extracted_metadata.csv`, `sib_extracted_metadata.csv`).  If all your files come from the same application there will be just one CSV.


In [30]:
extractor = MEI_Metadata_Extractor(verbose=True)
extractor.save_csvs(mei_input_folder, csv_output_folder)

Processing Ror0101.mei …
Processing Ror0102.mei …
Processing Ror0103.mei …
Processing Ror0104.mei …
Processing Ror0105.mei …
Processing Ror0106.mei …
Processing Ror0107.mei …
Saved 7 row(s) → extracted_metadata_csv/hum_drum_extracted_metadata.csv


### 2c. Understanding your extracted CSV

Open the CSV in a spreadsheet application.  You will see one row per MEI file and the following columns:

| Column | Contents |
|--------|----------|
| `filename` | The exact filename of the MEI file.  **Do not change this** — it is used to match rows back to files. |
| `source_type` | Detected application: `musescore`, `sibelius`, `humdrum`, or `mei_friend`. |
| `mei_version` | MEI schema version declared in the file. |
| `title` | Main title of the work. |
| `composer_name` | Composer's name as it appears in the file. |
| `composer_auth` | Authority system used (e.g. VIAF, GND). |
| `composer_auth_uri` | Full URI to the composer's authority record. |
| `editors` | Names and roles of editors/encoders (see below). |
| `encoding_date` | Date the file was encoded or last edited. |
| `rights` | Rights or availability statement. |
| `publisher` | Publisher name, if present. |
| `distributor` | Distributor(s), if present. |
| `genre` | Genre or classification term. |
| `encoding_application` | Software used to create the file. |
| `work_title` | Title as recorded in the workList section (may differ from the main title). |
| `movement_name` | Movement name (Humdrum files). |
| `source_title` | Title of the physical source edition (Humdrum files). |
| `source_editor` | Editor of the source edition (Humdrum files). |
| `source_encoder` | Person who encoded the file (Humdrum files). |
| `edition_version` | Version identifier of the edition (Humdrum files). |
| `encoding_annot` | Free-text note about the encoding (Humdrum files). |
| `humdrum_id` | Internal Humdrum identifier (Humdrum files). |

Columns that do not apply to a given source type are left blank.

---

#### How to record multiple editors or distributors

Use a **pipe character (`|`)** to separate multiple values in a single cell.  Each value will become its own element in the updated MEI file.

**Editors** can optionally include a role in square brackets:

```
Jane Smith [editor]|Robert Jones [encoder]|Maria Lopez [analyst]
```

If you omit the role brackets, the name is written without a role attribute.

**Distributors** (the `distributor` column) follow the same pattern:

```
Haverford College|CESR Tours|The CRIM Project
```

---

#### What happens if a cell is left empty?

**Nothing** — the existing content of that field in the MEI file is left exactly as it is.  You only need to fill in cells where you want to make a change.  This means you can apply the same CSV to update just one or two fields across a large corpus without touching anything else.

---

#### Saving your edited CSV

Save your edited CSV in any convenient location.  In the next step you will point the updater either to:
- a **local file path** (e.g. `updated_metadata/my_corpus_metadata.csv`), or
- a **URL** — Google Sheets and GitHub both work.

**Google Sheets:** File → Share → Publish to web → select the sheet → choose *Comma-separated values (.csv)* → copy the URL.

**GitHub:** navigate to the raw file view and copy the URL (it should start with `https://raw.githubusercontent.com/…`).


---
## 3. Apply Updated Metadata to Your MEI Files

Once you have edited your CSV, this step reads it and writes the new values into each MEI file.  Updated files are saved to a separate output folder with `_rev` added to their names (e.g. `MySong.mei` → `MySong_rev.mei`), so your originals are never overwritten.

> **Note:** Only non-empty cells are applied.  If you left a cell blank, the existing MEI content for that field is preserved unchanged.


### 3a. Specify folders and your updated CSV


In [34]:
# Folder containing the MEI files you want to update
#   (can be the same folder used in Step 2, or a different corpus)
mei_input_folder = 'mei_to_process'

# Your updated CSV — local path OR a URL (Google Sheets, GitHub raw)
csv_source = 'updated_metadata_csv_files/updated_metadata.csv'  # Example using a local file path

# Example using a Google Sheets URL:
# csv_source = 'https://docs.google.com/spreadsheets/d/e/YOURKEY/pub?output=csv'

# Example using a GitHub raw URL:
# csv_source = 'https://raw.githubusercontent.com/yourorg/yourrepo/main/metadata.csv'

# Folder where updated MEI files will be written
mei_output_folder = 'mei_with_updated_metadata'

In [35]:
csv_source

'updated_metadata_csv_files/updated_metadata.csv'

### 3b. Run the updater


In [36]:
updater = MEI_Metadata_Updater_Generic(verbose=True)
updater.process_folder(
    input_folder=mei_input_folder,
    csv_source=csv_source,
    output_folder=mei_output_folder
)

Loaded 7 row(s) from updated_metadata_csv_files/updated_metadata.csv
Updating Ror0101.mei …
  Saved Ror0101_rev.mei
Updating Ror0102.mei …
  Saved Ror0102_rev.mei
Updating Ror0103.mei …
  Saved Ror0103_rev.mei
Updating Ror0104.mei …
  Saved Ror0104_rev.mei
Updating Ror0105.mei …
  Saved Ror0105_rev.mei
Updating Ror0106.mei …
  Saved Ror0106_rev.mei
Updating Ror0107.mei …
  Saved Ror0107_rev.mei
Done. Updated 7 file(s), skipped 0.


#### What to expect

- A line is printed for each file that is updated.
- Files whose name does not appear in the CSV are skipped with a note.
- A summary at the end shows how many files were updated and how many were skipped.
- If a file cannot be parsed (e.g. corrupted XML), an error message is printed and processing continues with the next file.


---
## 4. CRIM Project Workflow

If you are working with the **Citations: The Renaissance Imitation Mass (CRIM) Project**, use this section instead of Sections 2–3.  The CRIM workflow uses a different set of column names that match the existing CRIM metadata spreadsheet, and it writes additional source-provenance fields (publisher, shelfmark, repository, etc.) into a `manifestationList` section of each MEI file.

The CRIM metadata columns are:

| Column | Contents |
|--------|----------|
| `MEI_Name` | Exact filename — **do not change** |
| `Title` | Work title |
| `Composer_Name` | Composer name |
| `Composer_VIAF` | Full VIAF URI for the composer |
| `Editor` | Pipe-separated editor names |
| `Copyright_Owner` | Pipe-separated copyright holders |
| `Rights_Statement` | Rights or license text |
| `Genre` | Genre |
| `Source_Title` | Title of the printed source edition |
| `Source_Publisher_1` | First publisher of the source |
| `Publisher_1_VIAF` | VIAF URI for first publisher |
| `Source_Date` | Publication date of the source |
| `Source_Institution` | Library or archive holding the source |
| `Source_Shelfmark` | Shelfmark / call number |
| `Source_Location` | City or location of the institution |
| `Source_Publisher_2` | Second publisher, if any |
| `Publisher_2_VIAF` | VIAF URI for second publisher |

Fields that are already present in your MEI files (from a previous CRIM processing run) will be extracted automatically.  Fields not yet in the files will be blank for you to fill in.


### 4a. Extract metadata in CRIM mode

All MEI files in the folder are written to a single CSV called `crim_extracted_metadata.csv`.


In [21]:
# Folder containing CRIM MEI files
mei_input_folder = 'mei_to_process'

# Folder where the CSV will be saved
csv_output_folder = 'sample_extracted_metadata_csv_files'

crim_extractor = MEI_Metadata_Extractor(verbose=True, crim_mode=True)
crim_extractor.save_csvs(mei_input_folder, csv_output_folder)

Processing Ror0101.mei …
Processing Ror0102.mei …


Processing Ror0103.mei …
Processing Ror0104.mei …
Processing Ror0105.mei …
Processing Ror0106.mei …
Processing Ror0107.mei …
Saved 7 row(s) → sample_extracted_metadata_csv_files/crim_extracted_metadata.csv


### 4b. Edit your CRIM metadata CSV

Open `crim_extracted_metadata.csv`, fill in or correct the columns, and save it — locally or to a shared spreadsheet.

The same rules apply as in Section 2c:
- Leave a cell blank to preserve the existing MEI value.
- Separate multiple editors with `|`: `André Vierendeels|Richard Freedman`.
- Separate multiple copyright owners with `|`: `Haverford College|CESR Tours`.


### 4c. Apply updated CRIM metadata


In [40]:
# Folder containing your CRIM MEI files
mei_input_folder = 'mei_to_process'

# Your updated CRIM CSV — local path or URL
# csv_source = 'updated_metadata_csv_files/updated_crim_metadata.csv'  # Example using a local file path

# Example: load directly from the shared CRIM Google Sheet
csv_source = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vSYa2WllTX_oOwDBI_is9hnqUGBzoksEj0YgvW_ns_nL04BuC7GdDN1IYoZLsIvjwjtkF8C6bVeSmXo/pub?output=csv'

# Output folder for updated files
mei_output_folder = 'mei_with_updated_metadata'

crim_updater = MEI_Metadata_Updater_Generic(verbose=True)
crim_updater.process_folder(
    input_folder=mei_input_folder,
    csv_source=csv_source,
    output_folder=mei_output_folder,
    crim_mode=True
)

Loaded 317 row(s) from https://docs.google.com/spreadsheets/d/e/2PACX-1vSYa2WllTX_oOwDBI_is9hnqUGBzoksEj0YgvW_ns_nL04BuC7GdDN1IYoZLsIvjwjtkF8C6bVeSmXo/pub?output=csv
  No CSV row for Ror0101.mei — skipping
  No CSV row for Ror0102.mei — skipping
  No CSV row for Ror0103.mei — skipping
  No CSV row for Ror0104.mei — skipping
  No CSV row for Ror0105.mei — skipping
  No CSV row for Ror0106.mei — skipping
  No CSV row for Ror0107.mei — skipping
Done. Updated 0 file(s), skipped 7.


---
## 5. Correcting Music Features

The `MEI_Music_Feature_Processor` corrects common encoding issues that arise when exporting MEI from notation software.  Each correction can be switched on or off independently.

Available options, with their default values:

| Option | Default | What it does |
|--------|---------|--------------|
| `resolve_multibar_ties` | `True` | Converts standalone `<tie>` elements spanning multiple measures into note-level `@tie` attributes (`i` / `m` / `t`) |
| `remove_incipit` | `True` | Removes the incipit measure (labeled 0 / n=1) and renumbers the remaining measures from 1 |
| `remove_incipit_leuven` | `False` | Removes Leuven-style incipits (leading invisible-barline measures); transfers meter from the following scoreDef to the first scoreDef |
| `remove_pb` | `True` | Removes all `<pb>` page-break elements |
| `remove_sb` | `True` | Removes all `<sb>` system-break elements |
| `remove_annotation` | `True` | Removes all `<annot>` annotation elements |
| `remove_ligature_bracket` | `True` | Removes `<bracketSpan>` ligature-bracket elements |
| `remove_dir` | `True` | Removes all `<dir>` direction / text-direction elements |
| `remove_variants` | `True` | Collapses `<app>` variant apparatus: retains the `<lem>` reading and discards all `<rdg>` alternatives |
| `remove_anchored_text` | `True` | Removes `<anchoredText>` elements |
| `remove_timestamp` | `True` | Strips `tstamp.real`, `vel`, `tstamp`, and `tstamp2` attributes from notes, rests, whole-measure rests, and ties |
| `remove_chord` | `True` | Removes `<chord>` wrapper elements |
| `check_for_chords` | `True` | Reports any remaining `<chord>` elements and the measure numbers where they appear (diagnostic only — does not remove) |
| `fix_elisions` | `True` | Corrects multi-syllable `<verse>` elements: concatenates elided syllables with `=` and sets proper `con` / `wordpos` attributes |
| `fix_musescore_elisions` | `True` | Fixes MuseScore-specific elision encoding: converts `con="b"` to `con="d"` and replaces combining breve characters with underscores |
| `slur_to_tie` | `True` | Re-tags `<slur>` elements as `<tie>` (also removes the `layer`, `tstamp`, `tstamp2`, and `staff` attributes that are invalid on ties) |
| `correct_ficta` | `True` | Converts colored (red) notes to editorial accidentals: wraps the accidental in `<supplied reason="edit">` and promotes `accid.ges` to `accid` |
| `voice_labels` | `True` | Moves `<label>` child-element text into the `@label` attribute of the enclosing `<staffDef>` |
| `correct_mrests` | `True` | Splits `<mRest>` whole-measure rests inside 3/1 measures into three individual whole rests |
| `collapse_layers` | `False` | Merges all voice layers into layer 1 |
| `remove_senfl_bracket` | `False` | Removes `<line type="bracket">` Senfl-edition bracket elements |
| `remove_empty_verse` | `False` | Removes `<verse>` elements that contain no children |
| `remove_lyrics` | `False` | Removes all `<verse>` elements (strips all lyrics from the file) |
| `correct_cmme_time_signatures` | `False` | Moves meter attributes from `<staffDef>` up to the enclosing `<scoreDef>` (fixes CMME export format) |
| `correct_jrp_time_signatures` | `False` | Promotes meter from `<meterSig>` child elements to `@meter.count` / `@meter.unit` attributes on `<scoreDef>` (fixes JRP export format) |


### 5a. Create an instance of the processor


In [23]:
music_feature_processor = MEI_Music_Feature_Processor()

### 5b. Specify input and output folders


In [37]:
# Input: folder of MEI files to process
# note that this can be the same folder used in previous steps, or a different corpus
mei_paths = glob.glob('mei_with_updated_metadata/*.mei')

# Output folder
output_folder = 'mei_with_updated_music_features'

In [38]:
mei_paths

['mei_with_updated_metadata/Ror0106_rev.mei',
 'mei_with_updated_metadata/Ror0107_rev.mei',
 'mei_with_updated_metadata/Ror0105_rev.mei',
 'mei_with_updated_metadata/Ror0104_rev.mei',
 'mei_with_updated_metadata/Ror0101_rev.mei',
 'mei_with_updated_metadata/Ror0102_rev.mei',
 'mei_with_updated_metadata/Ror0103_rev.mei']

### 5c. Process the files

Edit the `True` / `False` values below to control which corrections are applied.


In [39]:
for mei_path in mei_paths:
    music_feature_processor.process_music_features(
        mei_path,
        output_folder=output_folder,
        # --- on by default ---
        resolve_multibar_ties=True,
        remove_incipit=True,
        remove_pb=True,
        remove_sb=True,
        remove_annotation=True,
        remove_ligature_bracket=True,
        remove_dir=True,
        remove_variants=True,
        remove_anchored_text=True,
        remove_timestamp=True,
        remove_chord=True,
        check_for_chords=True,
        fix_elisions=True,
        fix_musescore_elisions=True,
        slur_to_tie=True,
        correct_ficta=True,
        voice_labels=True,
        correct_mrests=True,
        # --- off by default ---
        remove_incipit_leuven=False,
        collapse_layers=False,
        remove_senfl_bracket=False,
        remove_empty_verse=False,
        remove_lyrics=False,
        correct_cmme_time_signatures=False,
        correct_jrp_time_signatures=True,
    )

Getting Ror0106_rev
Found 0 page breaks to remove.
Found 0 section breaks to remove.
Found 1 annotations to remove.
  Tie resolution: 68 chains processed, 0 spanning 3+ measures.
Found 0 direction elements to remove.
Found 0 ligatures to remove.
Found 0 variants to correct.
Checking and Removing timestamp.
Found 1 scoreDef elements to process.
Found 0 3/1 measures check for mRests.
Corrected 0 mRests
Found 0 chord elements to remove.
Found elided syllables to correct.
Found elided syllables to correct.
Found elided syllables to correct.
Found elided syllables to correct.
Found elided syllables to correct.
Found elided syllables to correct.
Found elided syllables to correct.
Found elided syllables to correct.
Found elided syllables to correct.
Found elided syllables to correct.
Found elided syllables to correct.
Found elided syllables to correct.
Found elided syllables to correct.
Found elided syllables to correct.
Found elided syllables to correct.
Found elided syllables to correct.
Fo